In [15]:
import os
import json
import math
import csv
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications.efficientnet import preprocess_input

EXPORT_5CLS = r"C:\Users\Kuba\Desktop\praca_inz"
WEIGHTS     = os.path.join(EXPORT_5CLS, "best_stage2_weights.keras")
CLASS_JSON  = os.path.join(EXPORT_5CLS, "class_names.json")
PREDICTOR_PATH = r"C:\Users\Kuba\Desktop\praca_inz\shape_predictor_68_face_landmarks.dat"

with open(CLASS_JSON, "r", encoding="utf-8") as f:
    CLASS_NAMES = json.load(f)

IMG_SIZE = (224, 224)
base = EfficientNetB0(include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), weights=None)
x = GlobalAveragePooling2D()(base.output)
x = Dropout(0.3)(x)
out = Dense(len(CLASS_NAMES), activation='softmax')(x)

emo_model = Model(base.input, out)
emo_model.load_weights(WEIGHTS)

def predict_emotion_on_face(face_bgr):
    rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(rgb, IMG_SIZE)
    arr = img_to_array(img).astype(np.float32)
    arr = preprocess_input(arr)
    probs = emo_model.predict(np.expand_dims(arr, 0), verbose=0)[0]
    idx = int(np.argmax(probs))
    return CLASS_NAMES[idx], float(probs[idx]), probs

def get_head_pose(shape):
   
    nose = shape[30]
    left_eye = shape[36]
    right_eye = shape[45]
    
    # Wyznaczenie środka linii oczu
    eyes_center_x = (left_eye[0] + right_eye[0]) / 2.0
    eyes_center_y = (left_eye[1] + right_eye[1]) / 2.0
    
    face_width = np.linalg.norm(left_eye - right_eye) + 1e-6
    
    diff_x = nose[0] - eyes_center_x
    diff_y = nose[1] - eyes_center_y
    
    # Mnożymy przez -1, aby wynik na wykresie był zgodny z pozycją głowy człowieka
    vertical_score = -1 * (diff_y / face_width) * 100
    horizontal_score = (diff_x / face_width) * 100
    
    return vertical_score, horizontal_score, 0

BACKEND = None
try:
    import dlib
    from imutils import face_utils
    from scipy.spatial import distance

    detector  = dlib.get_frontal_face_detector()
    if os.path.exists(PREDICTOR_PATH):
        predictor = dlib.shape_predictor(PREDICTOR_PATH)
    else:
        predictor = None
        raise ImportError("Nie znaleziono pliku shape_predictor_68_face_landmarks.dat")


    LEFT_EYE  = slice(*face_utils.FACIAL_LANDMARKS_IDXS["left_eye"])
    RIGHT_EYE = slice(*face_utils.FACIAL_LANDMARKS_IDXS["right_eye"])
    MOUTH_IN  = slice(60, 68)

    def EAR(points6):
        p1,p2,p3,p4,p5,p6 = points6
        return (distance.euclidean(p2,p6) + distance.euclidean(p3,p5)) / (2.0*distance.euclidean(p1,p4) + 1e-8)

    def MAR(points8):
        p60,p61,p62,p63,p64,p65,p66,p67 = points8
        return (distance.euclidean(p63,p65) + distance.euclidean(p62,p66)) / (2.0*distance.euclidean(p60,p64) + 1e-8)

    def get_face_and_metrics(frame_bgr):
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        rects = detector(gray, 0)
        
        if len(rects) == 0 or predictor is None:
            return None
        
        rect = max(rects, key=lambda r: r.width()*r.height())
        shape = predictor(gray, rect)
        pts = face_utils.shape_to_np(shape)

        ear = (EAR(pts[LEFT_EYE]) + EAR(pts[RIGHT_EYE])) / 2.0
        mar = MAR(pts[MOUTH_IN])
        pitch, roll, _ = get_head_pose(pts)

        x1, y1, x2, y2 = rect.left(), rect.top(), rect.right(), rect.bottom()
        pad = 0.12
        w, h = x2-x1, y2-y1
        cx, cy = x1+w/2, y1+h/2
        x1 = int(max(0, cx - w*(1+pad)/2)); x2 = int(min(frame_bgr.shape[1]-1, cx + w*(1+pad)/2))
        y1 = int(max(0, cy - h*(1+pad)/2)); y2 = int(min(frame_bgr.shape[0]-1, cy + h*(1+pad)/2))
        face_crop = frame_bgr[y1:y2, x1:x2]
        
        return face_crop, ear, mar, pitch, roll

    BACKEND = "dlib"
    
except Exception as e:
    import mediapipe as mp
    BACKEND = "mediapipe"

def _smooth_np(x, win):
    if win is None or win <= 1 or len(x) == 0: return x
    k = np.ones(int(win), dtype=np.float32) / float(int(win))
    return np.convolve(np.asarray(x, dtype=np.float32), k, mode="same")

def process_video(video_path, out_csv, sample_every=1, smooth_win=9, write_preview=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    fcsv = open(out_csv, "w", newline="", encoding="utf-8")
    w = csv.writer(fcsv)
    
    header = ["frame","time_s","ear","mar", "pitch", "roll"] + [f"p_{c}" for c in CLASS_NAMES] + ["emo_top"]
    w.writerow(header)

    t0 = time.time()
    used = 0
    all_ear, all_mar = [], []
    emo_hist = {c:0 for c in CLASS_NAMES}

    preview_writer = None
    if write_preview:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        preview_writer = cv2.VideoWriter(write_preview, fourcc, fps/max(1, sample_every), (640, 360))

    fidx = -1
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        fidx += 1
        if fidx % sample_every != 0: continue

        met = get_face_and_metrics(frame)
        if met is None: continue
        face, ear, mar, pitch, roll_val = met
        
        if face.size == 0: continue

        label, prob, probs = predict_emotion_on_face(face)
        probs = [float(p) for p in probs]
        tsec = fidx / max(1.0, fps)
        
        row = [fidx, round(tsec,6), float(ear), float(mar), float(pitch), float(roll_val)] + probs + [label]
        w.writerow(row)

        all_ear.append(float(ear))
        all_mar.append(float(mar))
        emo_hist[label] += 1
        used += 1

        if preview_writer is not None:
            vis = cv2.resize(frame, (640,360))
            txt = f"EAR:{ear:.2f} MAR:{mar:.2f} V:{pitch:.1f} {label}"
            cv2.putText(vis, txt, (10,28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2, cv2.LINE_AA)
            preview_writer.write(vis)

    cap.release()
    if preview_writer is not None: preview_writer.release()
    fcsv.close()

    if smooth_win and smooth_win > 1:
        data = np.genfromtxt(out_csv, delimiter=',', names=True, dtype=None, encoding='utf-8')
        if data.size > 0:
            ear_s = _smooth_np(data['ear'], smooth_win)
            mar_s = _smooth_np(data['mar'], smooth_win)
            pitch_raw = data['pitch']
            roll_raw = data['roll']
            
            with open(out_csv, "w", newline="", encoding="utf-8") as f2:
                w2 = csv.writer(f2)
                w2.writerow(list(data.dtype.names))
                for i in range(len(data)):
                    row = list(data[i])
                    names = list(data.dtype.names)
                    row[names.index('ear')] = float(ear_s[i])
                    row[names.index('mar')] = float(mar_s[i])
                    row[names.index('pitch')] = float(pitch_raw[i])
                    row[names.index('roll')] = float(roll_raw[i])
                    w2.writerow(row)

    dur = time.time() - t0
    top_emo = max(emo_hist.items(), key=lambda x: x[1])[0] if used else None
    stats = {
        "backend": BACKEND,
        "fps": fps,
        "frames_total": total,
        "frames_used": used,
        "duration_s": round(dur, 3),
        "mean_ear": float(np.mean(all_ear)) if all_ear else None,
        "mean_mar": float(np.mean(all_mar)) if all_mar else None,
        "top_emotion": top_emo
    }
    return stats

def plot_timeseries(csv_path, out_png=None, show=True, y_ear=0.17, y_mar=0.27, min_event_len=3):
    try:
        data = np.genfromtxt(csv_path, delimiter=',', names=True, dtype=None, encoding='utf-8')
    except Exception: return
    if data.size == 0: raise RuntimeError("Empty CSV")

    f = data['frame']
    ear = data['ear']
    mar = data['mar']
    pitch = data['pitch'] 
    roll = data['roll']
    emo = data['emo_top']

    # Kalibracja głowy
    if len(pitch) > 0:
        pitch_zero = np.median(pitch)
        roll_zero = np.median(roll)
    else:
        pitch_zero = 0; roll_zero = 0

    pitch_centered = pitch - pitch_zero
    roll_centered = roll - roll_zero

    eye_state = np.where(ear < y_ear, 1, 0)
    eye_state_scaled = np.where(eye_state == 1, 0.35, 0.15) 
    events = []
    start = None
    for i in range(len(eye_state)):
        if eye_state[i] == 1 and start is None: start = i
        elif eye_state[i] == 0 and start is not None:
            if i - start >= min_event_len: events.append((f[start], f[i-1]))
            start = None
    if start is not None and len(eye_state) - start >= min_event_len:
        events.append((f[start], f[-1]))

    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, 
        sharex=True, 
        gridspec_kw={'height_ratios': [3, 2, 1]}, 
        figsize=(16, 12)
    )

    ax1.plot(f, ear, label="EAR", color='blue', zorder=3)
    ax1.plot(f, mar, label="MAR", color='orange', zorder=3)
    ax1.plot(f, eye_state_scaled, label="Stan Oka", color='purple', lw=0.8, alpha=0.4, zorder=1)
    
    ax1.axhline(y_ear, ls='--', lw=1.2, label=f'Próg EAR ({y_ear})')
    ax1.axhline(y_mar, ls=':', lw=1.2, label=f'Próg MAR ({y_mar})')
    
    ax1.set_ylabel("Wartość", fontsize=10)
    ax1.set_title("Analiza Zmęczenia (Oczy i Usta)", fontsize=12, pad=10)
    ax1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=5, frameon=False)

    for t1, t2 in events:
        ax1.axvspan(t1, t2, color='purple', alpha=0.2, zorder=1)
        mid = (t1 + t2) / 2; dur = t2 - t1 + 1
        ax1.text(mid, max(max(ear), max(mar)) * 1.05, f"{int(dur)}f", color='purple', ha='center', va='bottom', fontsize=8, zorder=4)

    ax2.plot(f, pitch_centered, label="Pitch (Pion)", color='green', lw=1.5)
    ax2.plot(f, roll_centered, label="Roll (Poziom)", color='red', lw=1.5, ls=':')
    
    ax2.axhline(-12, color='black', ls='--', alpha=0.5, label="Próg Kiwnięcia (-12%)")
    ax2.axhline(12, color='black', ls='--', alpha=0.2) 
    
    ax2.set_ylabel("Przesunięcie (%)", fontsize=10)
    ax2.set_title("Pozycja Głowy", fontsize=12, pad=10)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)

    emo_palette = {"Neutral": "#4C7B8A", "Happy": "#59A14F", "Sad": "#B07AA1", "Fear": "#F58518", "Angry": "#E45756"}
    ax3.set_xlim(f[0], f[-1]); ax3.set_ylim(0, 1); ax3.set_yticks([]); 
    ax3.set_title("Wykryta Emocja", fontsize=12, pad=10)
    ax3.set_xlabel("Klatka Wideo", fontsize=10)

    def _segments(tt, ll):
        start = 0
        for i in range(1, len(ll)):
            if ll[i] != ll[i-1]: yield (tt[start], tt[i], ll[i-1]); start = i
        yield (tt[start], tt[-1], ll[-1])
        
    for t1, t2, lab in _segments(f, emo):
        c = emo_palette.get(lab, "#999999")
        ax3.axvspan(t1, t2, color=c, alpha=0.95)
    
    handles = [plt.Line2D([0],[0], color=c, lw=10, label=k) for k,c in emo_palette.items()]
    ax3.legend(handles=handles, ncol=len(handles), loc="upper center", bbox_to_anchor=(0.5, -0.4), frameon=False)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.6)

    if out_png: os.makedirs(os.path.dirname(out_png), exist_ok=True); plt.savefig(out_png, bbox_inches="tight", dpi=150)
    if show: plt.show()
    plt.close(fig)

def create_driver_status_table(csv_path, fps, video_name="Unknown", save_file=None):
    ears, mars, pitches, emos = [], [], [], []
    try:
        with open(csv_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                try:
                    ears.append(float(row['ear'])); mars.append(float(row['mar']))
                    pitches.append(float(row.get('pitch', 0))); emos.append(row['emo_top'])
                except ValueError: continue 
    except FileNotFoundError: return "Error: CSV not found."

    total_frames = len(ears)
    if total_frames == 0: return "Error: Empty file."
    duration_sec = total_frames / fps

    # Ustawienie progów
    EAR_THRESH = 0.17  
    MAR_THRESH = 0.27
    PITCH_THRESH_NOD = -12.0 
    
    # Ustawienie minimalnego czasu trwania, aby zaliczyć do tabelki
    MIN_FRAMES_FATIGUE_EYE = int(fps * 0.3)     # 300ms / 9 klatek
    MIN_FRAMES_YAWN = int(fps * 0.5)            # 0.5s  / 15 klatek
    MIN_FRAMES_NOD = int(fps * 0.5)             # 0.5s  / 15 klatek
    

    p_raw = np.array(pitches)
    if len(p_raw) > 0: pitch_zero = np.median(p_raw)
    else: pitch_zero = 0
    pitch_centered = p_raw - pitch_zero
    
    yawn_details = []
    nod_details = []

    eyes_closed = (np.array(ears) < EAR_THRESH).astype(int)
    fatigue_eye_count = 0; current_streak = 0; total_closed_frames = 0
    for val in eyes_closed:
        if val == 1: current_streak += 1; total_closed_frames += 1
        else:
            if current_streak >= MIN_FRAMES_FATIGUE_EYE: fatigue_eye_count += 1
            current_streak = 0
    if current_streak >= MIN_FRAMES_FATIGUE_EYE: fatigue_eye_count += 1

    YAWN_EMOTIONS = {"Fear", "Angry", "Sad"} 
    yawn_count = 0; current_streak = 0; current_emotions = set(); start_frame = 0
    for i, (mar_val, emo_val) in enumerate(zip(mars, emos)):
        if mar_val > MAR_THRESH:
            if current_streak == 0: start_frame = i
            current_streak += 1; current_emotions.add(emo_val)
        else:
            if current_streak >= MIN_FRAMES_YAWN:
                if not current_emotions.isdisjoint(YAWN_EMOTIONS): 
                    yawn_count += 1
                    start_s = start_frame / fps
                    end_s = (i - 1) / fps
                    yawn_details.append({
                        "Start": start_frame, "End": i-1, 
                        "Frames": current_streak, 
                        "Start_Sec": round(start_s, 2), "End_Sec": round(end_s, 2)
                    })
            current_streak = 0; current_emotions = set()
            
    if current_streak >= MIN_FRAMES_YAWN:
        if not current_emotions.isdisjoint(YAWN_EMOTIONS): 
            yawn_count += 1
            start_s = start_frame / fps
            end_s = (total_frames - 1) / fps
            yawn_details.append({
                "Start": start_frame, "End": total_frames-1, 
                "Frames": current_streak, 
                "Start_Sec": round(start_s, 2), "End_Sec": round(end_s, 2)
            })

    nod_count = 0; current_streak = 0; start_frame = 0
    
    for i, p in enumerate(pitch_centered):
        if p < PITCH_THRESH_NOD: 
            if current_streak == 0: start_frame = i
            current_streak += 1
        else:
            if current_streak >= MIN_FRAMES_NOD: 
                nod_count += 1
                start_s = start_frame / fps
                end_s = (i - 1) / fps
                nod_details.append({
                    "Start": start_frame, "End": i-1, 
                    "Frames": current_streak, 
                    "Start_Sec": round(start_s, 2), "End_Sec": round(end_s, 2)
                })
            current_streak = 0
            
    if current_streak >= MIN_FRAMES_NOD: 
        nod_count += 1
        start_s = start_frame / fps
        end_s = (total_frames - 1) / fps
        nod_details.append({
            "Start": start_frame, "End": total_frames-1, 
            "Frames": current_streak,
            "Start_Sec": round(start_s, 2), "End_Sec": round(end_s, 2)
        })

    # Wskaźniki i wyniki
    perclos = (total_closed_frames / total_frames * 100)
    perclos_decimal = total_closed_frames / total_frames
    
    F_blink = fatigue_eye_count / duration_sec
    F_yawn = yawn_count / duration_sec
    F_nod = nod_count / duration_sec

    def normalize(x): return math.atan(x) * (2 / math.pi)
    Fatigue_Index = 0.5 * ( (0.5 * normalize(F_blink)) + (0.3 * normalize(F_yawn)) + (0.2 * normalize(F_nod)) + normalize(perclos_decimal) )

    emo_weights = {"Happy": -0.001, "Neutral": 0.000, "Angry": 0.002, "Sad": 0.002, "Fear": 0.003}
    total_emo_points = 0
    for emo in emos: total_emo_points += emo_weights.get(emo, 0.0)
    Emotion_Score = total_emo_points / duration_sec
    Comprehensive_Index = (0.5 * Emotion_Score) + (0.5 * Fatigue_Index)

    if Comprehensive_Index < 0.01: state = "Suitable for driving"
    elif 0.01 <= Comprehensive_Index < 0.02: state = "Lower risk"
    elif 0.02 <= Comprehensive_Index < 0.03: state = "Higher risk"
    else: state = "Unsuitable for driving"

    result = {
        "Video Name:": video_name,
        "Fatigue Eyes Closed Times": fatigue_eye_count,
        "Yawning Times": yawn_count,
        "Number of Drowsy Nods": nod_count,
        "PERCLOS (%)": round(perclos, 2),
        "Fatigue Comprehensive Indicators": round(Fatigue_Index, 4),
        "Emotions Score": round(Emotion_Score, 4),
        "Comprehensive Status Indicators": round(Comprehensive_Index, 4),
        "Predicted Driving States": state
    }

    print("\n" + "="*60)
    print("SZCZEGÓŁY ZDARZEŃ")
    print("="*60)
    
    if yawn_details:
        print(f"Wykryto {yawn_count} ziewnięć:")
        for idx, d in enumerate(yawn_details, 1):
            print(f"    {idx}. Trwało: {d['Frames']} klatek | Wystąpienie: {d['Start_Sec']}s - {d['End_Sec']}s")
    else:
        print("Brak ziewnięć.")

    if nod_details:
        print(f"\nWykryto {nod_count} kiwnięć głową (sennych):")
        for idx, d in enumerate(nod_details, 1):
            print(f"    {idx}. Trwało: {d['Frames']} klatek | Wystąpienie: {d['Start_Sec']}s - {d['End_Sec']}s")
    else:
        print("\nBrak kiwnięć głową.")

    if save_file:
        df_new = pd.DataFrame([result])
        if os.path.exists(save_file):
            try:
                df_existing = pd.read_excel(save_file)
                pd.concat([df_existing, df_new], ignore_index=True).to_excel(save_file, index=False)
                print(f"\nWynik dopisano do: {save_file}")
            except Exception as e: print(f"Błąd zapisu: {e}")
        else:
            df_new.to_excel(save_file, index=False)
            print(f"\nUtworzono nowy plik: {save_file}")

    return result

In [ ]:
import os
out_dir = r"C:\Users\Kuba\Desktop\praca_inz\YawDD\Dash\_out"
os.makedirs(out_dir, exist_ok=True)

summary_file = os.path.join(out_dir, "tabela_wynikow.xlsx")

video_paths_list = [
    r"C:\Users\Kuba\Desktop\praca_inz\YawDD\Dash\Dash\Male\4-MaleNoGlasses.avi",
    r"C:\Users\Kuba\Desktop\praca_inz\YawDD\Dash\Dash\Male\11-MaleGlasses.avi",
    r"C:\Users\Kuba\Desktop\praca_inz\YawDD\Dash\Dash\Female\5-FemaleNoGlasses.avi",
    r"C:\Users\Kuba\Desktop\praca_inz\YawDD\Dash\Dash\Female\13-FemaleGlasses.avi.avi",
    r"C:\Users\Kuba\Desktop\praca_inz\YawDD\Dash\Dash\Female\12-FemaleGlasses.avi.avi",
    r"C:\Users\Kuba\Desktop\praca_inz\YawDD\Dash\Dash\Male\13-MaleNoGlasses .avi"
]

print(f"Do przetworzenia {len(video_paths_list)} filmów")

for video_path in video_paths_list:
    if not os.path.exists(video_path):
        print(f"Nie znaleziono pliku: {video_path}")
        continue

    filename = os.path.basename(video_path)
    base_name = filename.replace(".avi", "")
    
    print(f"\n>>> PRZETWARZANIE: {filename}")
    
    csv_path = os.path.join(out_dir, f"{base_name}_dane.csv")
    png_path = os.path.join(out_dir, f"{base_name}_wykres.png")

    stats = process_video(video_path, csv_path, sample_every=1, smooth_win=9, write_preview=None)
    
    plot_timeseries(csv_path, out_png=png_path, show=True, y_ear=0.17, y_mar=0.27)

    row_data = create_driver_status_table(csv_path, stats['fps'], video_name=base_name, save_file=summary_file)
    print("\n" + "="*50)
    print("WYNIKI KOŃCOWE ANALIZY KIEROWCY:")
    print("="*50)
    for key, value in row_data.items():
        print(f"{key:40} : {value}")
    print("="*50)

print("\n" + "="*40)
print("KONIEC PRZETWARZANIA")
print(f"Wyniki w pliku: {summary_file}")
print("="*40)


--- START: Do przetworzenia 6 filmów ---

>>> PRZETWARZANIE: 4-MaleNoGlasses.avi

SZCZEGÓŁY ZDARZEŃ
Wykryto 3 ziewnięć:
    1. Trwało: 31 klatek | Wystąpienie: 11.53s - 12.53s
    2. Trwało: 91 klatek | Wystąpienie: 27.7s - 30.7s
    3. Trwało: 108 klatek | Wystąpienie: 49.13s - 52.7s

Brak kiwnięć głową.

Utworzono nowy plik: C:\Users\Kuba\Desktop\praca_inz\YawDD\Dash\_out\PODSUMOWANIE_WYNIKOW.xlsx

WYNIKI KOŃCOWE ANALIZY KIEROWCY:
Video Name:                              : 4-MaleNoGlasses
Fatigue Eyes Closed Times                : 3
Yawning Times                            : 3
Number of Drowsy Nods                    : 0
PERCLOS (%)                              : 1.84
Fatigue Comprehensive Indicators         : 0.0176
Emotions Score                           : 0.0188
Comprehensive Status Indicators          : 0.0182
Predicted Driving States                 : Lower risk

>>> PRZETWARZANIE: 11-MaleGlasses.avi

SZCZEGÓŁY ZDARZEŃ
Wykryto 2 ziewnięć:
    1. Trwało: 81 klatek | Wystąpienie: